# 01. Dataset Cleaning and Initial Inspection

This notebook loads a public UPI transaction dataset, inspects its schema, and performs the required cleaning steps before modeling.

In [4]:
from pathlib import Path
import sys
import pandas as pd
sys.path.append(str(Path.cwd().parent))
from src.upi_fraud_pipeline import find_dataset_path, load_dataset, dataset_report, infer_target_column, clean_dataframe

dataset_path = find_dataset_path()
print(f'Using dataset: {dataset_path}')
df, _ = load_dataset(dataset_path.name)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

Using dataset: D:\Shivam\UPI Fraud Detection Model\data\raw\upi_transactions_2024.csv
Shape: (300, 9)
Columns: ['transaction_id', 'timestamp', 'amount', 'merchant_category', 'transaction_type', 'device_type', 'network_type', 'sender_state', 'fraud']


,transaction_id,timestamp,amount,merchant_category,transaction_type,device_type,network_type,sender_state,fraud
0,TXN0,2024-01-01 00:00:00,2.236064,Bills,UPI,iPhone,UPI,TN,0
1,TXN1,2024-01-01 01:00:00,0.582758,Travel,Merchant,Web,IMPS,GJ,0
2,TXN2,2024-01-01 02:00:00,3.491918,Travel,Bank Transfer,Android,IMPS,MH,0
3,TXN3,2024-01-01 03:00:00,4.223080,Travel,QR,Web,IMPS,GJ,0
4,TXN4,2024-01-01 04:00:00,0.234328,Shopping,QR,Android,IMPS,TN,0


In [5]:
df.info()
df.describe(include='all').transpose()
df.isnull().sum()
df.nunique()
df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   transaction_id     300 non-null    object 
 1   timestamp          300 non-null    object 
 2   amount             300 non-null    float64
 3   merchant_category  300 non-null    object 
 4   transaction_type   300 non-null    object 
 5   device_type        300 non-null    object 
 6   network_type       300 non-null    object 
 7   sender_state       300 non-null    object 
 8   fraud              300 non-null    int64  
dtypes: float64(1), int64(1), object(7)
memory usage: 21.2+ KB


np.int64(0)

In [6]:
target_column = infer_target_column(df)
print('Target column:', target_column)
cleaned = clean_dataframe(df, target_column=target_column)
cleaned.head()
cleaned.to_csv(Path.cwd().parent / 'data' / 'processed' / 'cleaned_upi_dataset.csv', index=False)

Target column: fraud


## Cleaning decisions

- Missing values were handled according to data type.
- Duplicate rows were removed.
- Numeric values were coerced and invalid amounts were treated as missing.
- Date and time columns were converted to datetime where possible.
- The fraud target was normalized to binary values, 0 = Genuine and 1 = Fraud.